In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Atualizado: Caminho para a pasta data2 e arquivos específicos (1 a 4)
caminho_arquivos = "data2"
arquivos = [os.path.join(caminho_arquivos, f"reduced_data_{x}.csv") for x in range(1, 5)]

print(f"Lendo {len(arquivos)} arquivos CSV da pasta '{caminho_arquivos}'...")

# low_memory=False evita problemas de tipagem mista durante a leitura
df_list = [pd.read_csv(f, low_memory=False) for f in arquivos]
df_full = pd.concat(df_list, ignore_index=True)

print("Ajustando labels...")
df_full.rename(columns={'attack': 'label', 'category': 'attack_cat'}, inplace=True)

# 2. REMOÇÃO CIRÚRGICA: Evita a explosão do One-Hot Encoding
colunas_para_remover = [
    'pkSeqID', 'subcategory', 'saddr', 'daddr', 
    'sport', 'dport', # <-- Culpados pelo alto consumo de RAM
    'state_number', 'proto_number', 'flgs_number' # Redundantes
]
df_full.drop(columns=colunas_para_remover, inplace=True, errors='ignore')

# 3. REDUÇÃO DE TAMANHO (Opcional, mas RECOMENDADO PARA A WISARD)
# Se os arquivos 'reduced_data' já forem pequenos o suficiente, você pode 
# comentar esta linha. Se ainda tiverem milhões de linhas, mantenha-a.
print("Amostrando 10% do dataset para otimizar uso de RAM...")
df_full = df_full.sample(frac=0.10, random_state=42)

# 4. DIVISÃO E EXPORTAÇÃO
print("Dividindo em treino e teste...")
df_train, df_test = train_test_split(
    df_full, 
    test_size=0.30, 
    random_state=42, 
    stratify=df_full['attack_cat']
)

# Salva na pasta 'data' que é onde seu notebook original vai buscar os arquivos
os.makedirs("data2", exist_ok=True)
df_train.to_csv("data2/BotIoT_training-set.csv", index=False)
df_test.to_csv("data2/BotIoT_testing-set.csv", index=False)

print("\nFeito! Dataset limpo e pronto.")
print(f"Shape do Treino: {df_train.shape}")
print(f"Shape do Teste: {df_test.shape}")

Lendo 4 arquivos CSV da pasta 'data2'...
Ajustando labels...
Amostrando 10% do dataset para otimizar uso de RAM...
Dividindo em treino e teste...

Feito! Dataset limpo e pronto.
Shape do Treino: (256796, 37)
Shape do Teste: (110056, 37)
